# Chapter 4: Rotations in 3D

<a href="../lite/lab/index.html?path=ch04_rotations_3d.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# ── Rotation matrix builders ─────────────────────────────────────────────────
def Rx(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def Ry(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def Rz(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

# ── ZYX Euler angles to rotation matrix (roll-pitch-yaw) ─────────────────────
def euler_to_R(roll, pitch, yaw):
    return Rz(yaw) @ Ry(pitch) @ Rx(roll)

# ── Rotation matrix to ZYX Euler angles ──────────────────────────────────────
def R_to_euler(R):
    pitch = np.arctan2(-R[2, 0], np.sqrt(R[0, 0]**2 + R[1, 0]**2))
    if np.abs(np.cos(pitch)) > 1e-6:
        roll = np.arctan2(R[2, 1], R[2, 2])
        yaw = np.arctan2(R[1, 0], R[0, 0])
    else:  # gimbal lock
        roll = 0.0
        yaw = np.arctan2(-R[0, 1], R[1, 1])
    return roll, pitch, yaw

# ── Axis-angle to rotation matrix (Rodrigues) ────────────────────────────────
def axis_angle_to_R(axis, angle):
    axis = axis / np.linalg.norm(axis)
    K = np.array([[0, -axis[2], axis[1]],
                  [axis[2], 0, -axis[0]],
                  [-axis[1], axis[0], 0]])
    return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * K @ K

# ── Rotation matrix to axis-angle ────────────────────────────────────────────
def R_to_axis_angle(R):
    angle = np.arccos(np.clip((np.trace(R) - 1) / 2, -1, 1))
    if np.abs(angle) < 1e-10:
        return np.array([0, 0, 1]), 0.0
    if np.abs(angle - np.pi) < 1e-10:
        # Find eigenvector with eigenvalue 1
        vals, vecs = np.linalg.eig(R)
        idx = np.argmin(np.abs(vals - 1.0))
        axis = np.real(vecs[:, idx])
        return axis / np.linalg.norm(axis), angle
    axis = np.array([R[2,1]-R[1,2], R[0,2]-R[2,0], R[1,0]-R[0,1]]) / (2*np.sin(angle))
    return axis / np.linalg.norm(axis), angle

# ── Quaternion operations (q = [w, x, y, z]) ─────────────────────────────────
def quat_normalize(q):
    return q / np.linalg.norm(q)

def quat_multiply(q1, q2):
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2
    return np.array([w1*w2 - x1*x2 - y1*y2 - z1*z2,
                     w1*x2 + x1*w2 + y1*z2 - z1*y2,
                     w1*y2 - x1*z2 + y1*w2 + z1*x2,
                     w1*z2 + x1*y2 - y1*x2 + z1*w2])

def quat_conjugate(q):
    return np.array([q[0], -q[1], -q[2], -q[3]])

def quat_rotate(q, v):
    p = np.array([0, v[0], v[1], v[2]])
    result = quat_multiply(quat_multiply(q, p), quat_conjugate(q))
    return result[1:]

def quat_from_axis_angle(axis, angle):
    axis = axis / np.linalg.norm(axis)
    return np.array([np.cos(angle/2), *(np.sin(angle/2) * axis)])

def quat_to_R(q):
    q = quat_normalize(q)
    w, x, y, z = q
    return np.array([
        [1-2*(y*y+z*z), 2*(x*y-w*z),   2*(x*z+w*y)],
        [2*(x*y+w*z),   1-2*(x*x+z*z), 2*(y*z-w*x)],
        [2*(x*z-w*y),   2*(y*z+w*x),   1-2*(x*x+y*y)]])

def quat_from_R(R):
    w = 0.5 * np.sqrt(max(1 + R[0,0] + R[1,1] + R[2,2], 0))
    if w > 1e-6:
        x = (R[2,1] - R[1,2]) / (4*w)
        y = (R[0,2] - R[2,0]) / (4*w)
        z = (R[1,0] - R[0,1]) / (4*w)
    else:
        x = np.sqrt(max((R[0,0]+1)/2, 0))
        y = np.sqrt(max((R[1,1]+1)/2, 0))
        z = np.sqrt(max((R[2,2]+1)/2, 0))
        # Fix signs
        if R[0,1] + R[1,0] < 0: y = -y
        if R[0,2] + R[2,0] < 0: z = -z
    return quat_normalize(np.array([w, x, y, z]))

def slerp(q0, q1, t):
    q0, q1 = quat_normalize(q0), quat_normalize(q1)
    dot = np.dot(q0, q1)
    if dot < 0:  # take shorter path
        q1, dot = -q1, -dot
    dot = np.clip(dot, -1, 1)
    omega = np.arccos(dot)
    if omega < 1e-10:
        return quat_normalize((1-t)*q0 + t*q1)
    return (np.sin((1-t)*omega)*q0 + np.sin(t*omega)*q1) / np.sin(omega)

# ── Helper: draw a 3D frame ──────────────────────────────────────────────────
def draw_frame_3d(ax, R=np.eye(3), origin=np.zeros(3), label="", length=1.0):
    colors = ['red', 'green', 'blue']
    labels_ax = ['x', 'y', 'z']
    for i, (c, la) in enumerate(zip(colors, labels_ax)):
        v = R[:, i] * length
        ax.quiver(*origin, *v, color=c, arrow_length_ratio=0.15, lw=2)
    if label:
        ax.text(origin[0], origin[1], origin[2]-length*0.3, label,
                fontsize=10, fontweight='bold', ha='center')

## 4.1 Euler Angles and Limitations

In 3D, we can build any rotation from three successive rotations about coordinate axes.
The most common convention in robotics is **ZYX** (yaw-pitch-roll):

$$R(\phi, \theta, \psi) = R_z(\psi) \; R_y(\theta) \; R_x(\phi)$$

where $\phi$ = roll, $\theta$ = pitch, $\psi$ = yaw.

### Example: Rotating a drone

A quadrotor's orientation is typically described by roll, pitch, and yaw.
Let's visualize how these three angles combine.

```{admonition} What you will build
:class: tip

- Rotate 3D objects using Euler angles, axis-angle, and quaternions
- Demonstrate gimbal lock and explain why it breaks Euler angle based systems
- Interpolate smoothly between two orientations using quaternion SLERP
- Rotate a 3D LiDAR point cloud using quaternion arithmetic

**Real world application:** Drones, robotic arms, and VR headsets all represent orientation in 3D. After this chapter, you will know which representation to use and why most production systems choose quaternions.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **Eigen::Quaterniond** | C++ quaternion class used in most robotics software |
| **scipy.spatial.transform.Rotation** | Python class for converting between Euler, quaternion, rotation matrix, and axis-angle |
| **tf_transformations (ROS)** | ROS utility for quaternion math, Euler conversions, and SLERP |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
roll_deg = 15.0     # roll (rotation about x)   (try 0, 30, -20)
pitch_deg = 25.0    # pitch (rotation about y)  (try 0, 45, 89)
yaw_deg = 40.0      # yaw (rotation about z)    (try 0, 90, 180)
# ──────────────────────────────────────────────────────────────────────────────

roll, pitch, yaw = np.radians(roll_deg), np.radians(pitch_deg), np.radians(yaw_deg)
R = euler_to_R(roll, pitch, yaw)

# Drone body: a cross shape
body_pts = np.array([[1,0,0],[-1,0,0],[0,1,0],[0,-1,0],
                      [0.8,0,0],[0.8,0.15,0.05],[-0.8,0,0],[-0.8,0.15,0.05],
                      [0,0.8,0],[0.15,0.8,0.05],[0,-0.8,0],[0.15,-0.8,0.05]]).T
body_rotated = R @ body_pts

fig = plt.figure(figsize=(14, 6))

# Original
ax1 = fig.add_subplot(121, projection='3d')
ax1.set_title("Original orientation", fontsize=13)
draw_frame_3d(ax1, np.eye(3), label="body", length=0.6)
# Draw drone arms
for arm in [([1,0,0],[-1,0,0]), ([0,1,0],[0,-1,0])]:
    p = np.array(arm).T
    ax1.plot3D(p[0], p[1], p[2], 'steelblue', lw=4)
# Rotors
for pos in [[1,0,0],[-1,0,0],[0,1,0],[0,-1,0]]:
    theta_c = np.linspace(0, 2*np.pi, 30)
    cx = pos[0] + 0.25*np.cos(theta_c)
    cy = pos[1] + 0.25*np.sin(theta_c)
    cz = np.full_like(theta_c, pos[2])
    ax1.plot3D(cx, cy, cz, 'gray', alpha=0.5)

# Rotated
ax2 = fig.add_subplot(122, projection='3d')
ax2.set_title(f"Roll={roll_deg:.0f}° Pitch={pitch_deg:.0f}° Yaw={yaw_deg:.0f}°", fontsize=13)
draw_frame_3d(ax2, R, label="body", length=0.6)
for arm in [([1,0,0],[-1,0,0]), ([0,1,0],[0,-1,0])]:
    p = R @ np.array(arm).T
    ax2.plot3D(p[0], p[1], p[2], 'tomato', lw=4)
for pos in [[1,0,0],[-1,0,0],[0,1,0],[0,-1,0]]:
    theta_c = np.linspace(0, 2*np.pi, 30)
    circle = np.array([pos[0] + 0.25*np.cos(theta_c),
                       pos[1] + 0.25*np.sin(theta_c),
                       np.full_like(theta_c, pos[2])])
    circle_r = R @ circle
    ax2.plot3D(circle_r[0], circle_r[1], circle_r[2], 'gray', alpha=0.5)

for ax in [ax1, ax2]:
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.set_zlim(-1.5, 1.5)
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")

plt.tight_layout()
plt.show()

### Gimbal lock

When pitch $= \pm 90°$, roll and yaw rotate about the **same axis**, we lose one degree of freedom.
This is called **gimbal lock**.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
pitch_test_deg = 90.0   # try 0, 45, 89, 90 to see gimbal lock appear
n_samples = 12          # number of orientation samples to show
# ──────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6),
                         subplot_kw={'projection': '3d'})

# Normal case: pitch = 0
ax = axes[0]; ax.set_title("Pitch = 0° (all 3 DOF)", fontsize=13)
for i in range(n_samples):
    roll_i = np.radians(i * 360 / n_samples)
    yaw_i = np.radians(i * 180 / n_samples)
    R_i = euler_to_R(roll_i, 0, yaw_i)
    v = R_i @ np.array([1, 0, 0])
    ax.quiver(0, 0, 0, v[0], v[1], v[2], color=plt.cm.viridis(i/n_samples),
              alpha=0.7, arrow_length_ratio=0.1)
ax.set_xlim(-1.5,1.5); ax.set_ylim(-1.5,1.5); ax.set_zlim(-1.5,1.5)
ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")

# Gimbal lock: pitch = 90°
ax = axes[1]; ax.set_title(f"Pitch = {pitch_test_deg:.0f}° — gimbal lock!", fontsize=13)
for i in range(n_samples):
    roll_i = np.radians(i * 360 / n_samples)
    yaw_i = np.radians(i * 180 / n_samples)
    R_i = euler_to_R(roll_i, np.radians(pitch_test_deg), yaw_i)
    v = R_i @ np.array([1, 0, 0])
    ax.quiver(0, 0, 0, v[0], v[1], v[2], color=plt.cm.magma(i/n_samples),
              alpha=0.7, arrow_length_ratio=0.1)
ax.set_xlim(-1.5,1.5); ax.set_ylim(-1.5,1.5); ax.set_zlim(-1.5,1.5)
ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")

plt.tight_layout()
plt.show()

# Demonstrate numerically
print("Gimbal lock demonstration (pitch = 90°):")
print("Varying roll and yaw produce the SAME rotations:\n")
for r, y in [(0, 30), (10, 20), (20, 10), (30, 0)]:
    R_test = euler_to_R(np.radians(r), np.radians(90), np.radians(y))
    print(f"  roll={r:3d}°, yaw={y:3d}° → R[0,0]={R_test[0,0]:.3f}, "
          f"R[0,1]={R_test[0,1]:.3f}, R[0,2]={R_test[0,2]:.3f}")

**Key observations:**
- At pitch $= 90°$, only the **sum** (or difference) of roll and yaw matters, you cannot control them independently.
- In the left plot, varying roll and yaw covers a hemisphere. In the right plot, all arrows lie on a **circle**, one DOF is lost.
- Gimbal lock is not a software bug; it is a fundamental limitation of representing 3D rotations with three angles.

## 4.2 Axis-Angle Representation

Any rotation can be described by a **unit axis** $\hat{k}$ and an **angle** $\theta$:

$$R(\hat{k}, \theta) = I \cos\theta + (1 - \cos\theta)\hat{k}\hat{k}^T + \sin\theta [\hat{k}]_\times$$

This is **Rodrigues' formula**. The skew-symmetric matrix $[\hat{k}]_\times$ is:

$$[\hat{k}]_\times = \begin{bmatrix} 0 & -k_z & k_y \\ k_z & 0 & -k_x \\ -k_y & k_x & 0 \end{bmatrix}$$

### Example: Rotating a point about an arbitrary axis

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
axis = np.array([1.0, 1.0, 1.0])    # rotation axis (will be normalized)  (try [0,0,1], [1,0,0])
angle_deg = 60.0                     # rotation angle (degrees)
point = np.array([1.0, 0.0, 0.0])   # point to rotate
# ──────────────────────────────────────────────────────────────────────────────

axis_n = axis / np.linalg.norm(axis)
angle = np.radians(angle_deg)
R = axis_angle_to_R(axis_n, angle)
point_rotated = R @ point

# Generate the arc of rotation
arc_angles = np.linspace(0, angle, 50)
arc_points = np.array([axis_angle_to_R(axis_n, a) @ point for a in arc_angles])

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Draw rotation axis
ax_line = np.array([-axis_n * 1.5, axis_n * 1.5])
ax.plot3D(ax_line[:, 0], ax_line[:, 1], ax_line[:, 2], 'k--', lw=1.5, alpha=0.5, label='rotation axis')

# Draw frame
draw_frame_3d(ax, np.eye(3), length=0.5)

# Original point
ax.scatter(*point, color='steelblue', s=80, zorder=5)
ax.text(*point + 0.1, "  original", color='steelblue', fontsize=11)

# Rotated point
ax.scatter(*point_rotated, color='tomato', s=80, zorder=5)
ax.text(*point_rotated + 0.1, "  rotated", color='tomato', fontsize=11)

# Arc
ax.plot3D(arc_points[:, 0], arc_points[:, 1], arc_points[:, 2], 'orange', lw=2.5, label='rotation arc')

# Axis label
ax.text(*(axis_n * 1.6), f"$\\hat{{k}}$ = [{axis_n[0]:.2f}, {axis_n[1]:.2f}, {axis_n[2]:.2f}]",
        fontsize=10)

ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.set_zlim(-1.5, 1.5)
ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
ax.set_title(f"Axis-angle rotation: {angle_deg:.0f}° about $\\hat{{k}}$", fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

# Verify round-trip
axis_recovered, angle_recovered = R_to_axis_angle(R)
print(f"Original:  axis = {axis_n}, angle = {angle_deg:.1f}°")
print(f"Recovered: axis = {np.array2string(axis_recovered, precision=3)}, "
      f"angle = {np.degrees(angle_recovered):.1f}°")

**Key observations:**
- Axis-angle is **intuitive**: "rotate this much about this direction."
- It has **no gimbal lock**, any orientation can be represented.
- The representation is singular at $\theta = 0$ (axis is undefined for zero rotation) and at $\theta = \pi$ (axis sign ambiguity).
- Composing two axis-angle rotations is not straightforward, you'd convert to matrices or quaternions first.

## 4.3 Quaternions

A **unit quaternion** $q = [w, x, y, z]$ with $\|q\| = 1$ represents a rotation.
It relates to axis-angle by:

$$q = \left[\cos\frac{\theta}{2}, \; \sin\frac{\theta}{2}\hat{k}\right]$$

**Key operations:**
- **Composition:** $q_{AC} = q_{AB} \otimes q_{BC}$ (quaternion multiplication)
- **Inverse:** $q^{-1} = q^* = [w, -x, -y, -z]$ (conjugate, since $\|q\| = 1$)
- **Rotate a vector:** $\mathbf{v}' = q \otimes [0, \mathbf{v}] \otimes q^*$

### Example: Quaternion rotation of a 3D object

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
axis = np.array([0.0, 0.0, 1.0])   # rotation axis   (try [1,0,0], [1,1,0])
angle_deg = 45.0                    # rotation angle   (try 30, 90, 120)
# ──────────────────────────────────────────────────────────────────────────────

q = quat_from_axis_angle(axis / np.linalg.norm(axis), np.radians(angle_deg))

# Create an L-shaped 3D object (vertices)
verts = np.array([[0,0,0],[2,0,0],[2,0.5,0],[0.5,0.5,0],[0.5,1.5,0],[0,1.5,0],
                  [0,0,0.5],[2,0,0.5],[2,0.5,0.5],[0.5,0.5,0.5],[0.5,1.5,0.5],[0,1.5,0.5]])

# Rotate each vertex
verts_rotated = np.array([quat_rotate(q, v) for v in verts])

# Also compute using rotation matrix to verify
R_from_q = quat_to_R(q)
verts_R = (R_from_q @ verts.T).T

fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(121, projection='3d')
ax1.set_title("Original + Quaternion rotated", fontsize=13)
draw_frame_3d(ax1, np.eye(3), length=0.5)

# Draw edges of L-shape
edges = [(0,1),(1,2),(2,3),(3,4),(4,5),(5,0),  # bottom face
         (6,7),(7,8),(8,9),(9,10),(10,11),(11,6),  # top face
         (0,6),(1,7),(2,8),(3,9),(4,10),(5,11)]  # vertical
for i, j in edges:
    ax1.plot3D(*zip(verts[i], verts[j]), 'steelblue', lw=1.5, alpha=0.5)
    ax1.plot3D(*zip(verts_rotated[i], verts_rotated[j]), 'tomato', lw=2)

ax1.set_xlim(-2, 2.5); ax1.set_ylim(-1, 2.5); ax1.set_zlim(-1, 2)
ax1.set_xlabel("X"); ax1.set_ylabel("Y"); ax1.set_zlabel("Z")

# Verify quaternion == matrix
ax2 = fig.add_subplot(122)
ax2.axis('off')
max_err = np.max(np.abs(verts_rotated - verts_R))
info = (f"Quaternion q = [{q[0]:.4f}, {q[1]:.4f}, {q[2]:.4f}, {q[3]:.4f}]\n"
        f"||q|| = {np.linalg.norm(q):.6f}\n\n"
        f"Rotation matrix from q:\n{np.array2string(R_from_q, precision=4, suppress_small=True)}\n\n"
        f"Max difference (quat vs matrix): {max_err:.2e}\n\n"
        f"Double cover: q and -q give the same rotation:\n"
        f"  R(q)  = R(-q)? {np.allclose(quat_to_R(q), quat_to_R(-q))}")
ax2.text(0.05, 0.5, info, fontsize=12, family='monospace',
         verticalalignment='center', transform=ax2.transAxes,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax2.set_title("Quaternion properties", fontsize=13)

plt.tight_layout()
plt.show()

**Key observations:**
- Quaternions have **no gimbal lock** and are **numerically stable**.
- $q$ and $-q$ represent the **same rotation** (double cover of SO(3)).
- Quaternion multiplication is more efficient than matrix multiplication (16 multiplications vs 27).
- The unit-norm constraint ($\|q\| = 1$) is easy to maintain, just normalize after each update.

## 4.4 Conversion Between Representations

All representations describe the same rotation, we can convert freely:

| From → To | Method |
|-----------|--------|
| Euler → Matrix | $R = R_z(\psi) R_y(\theta) R_x(\phi)$ |
| Matrix → Euler | Extract angles via $\text{atan2}$ (watch for gimbal lock) |
| Axis-angle → Matrix | Rodrigues' formula |
| Matrix → Axis-angle | $\theta = \arccos\frac{\text{tr}(R)-1}{2}$, axis from skew part |
| Quaternion ↔ Axis-angle | $q = [\cos\frac{\theta}{2}, \sin\frac{\theta}{2}\hat{k}]$ |
| Quaternion ↔ Matrix | Direct formulas (see setup cell) |

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
roll_deg = 25.0
pitch_deg = 40.0
yaw_deg = -15.0
# ──────────────────────────────────────────────────────────────────────────────

roll, pitch, yaw = np.radians(roll_deg), np.radians(pitch_deg), np.radians(yaw_deg)

# Forward: Euler → R → axis-angle → quaternion
R = euler_to_R(roll, pitch, yaw)
ax_recovered, ang_recovered = R_to_axis_angle(R)
q = quat_from_R(R)

# Round-trip: quaternion → R → Euler
R_from_q = quat_to_R(q)
roll2, pitch2, yaw2 = R_to_euler(R_from_q)

print("═" * 60)
print("  All representations of the SAME rotation")
print("═" * 60)
print(f"\n  Euler (ZYX):     roll={roll_deg:.1f}°  pitch={pitch_deg:.1f}°  yaw={yaw_deg:.1f}°")
print(f"\n  Rotation matrix:\n{np.array2string(R, precision=4, suppress_small=True)}")
print(f"\n  Axis-angle:      axis=[{ax_recovered[0]:.3f}, {ax_recovered[1]:.3f}, "
      f"{ax_recovered[2]:.3f}]  angle={np.degrees(ang_recovered):.1f}°")
print(f"\n  Quaternion:       [{q[0]:.4f}, {q[1]:.4f}, {q[2]:.4f}, {q[3]:.4f}]  "
      f"(||q||={np.linalg.norm(q):.6f})")
print(f"\n  Round-trip Euler: roll={np.degrees(roll2):.1f}°  pitch={np.degrees(pitch2):.1f}°  "
      f"yaw={np.degrees(yaw2):.1f}°")
print(f"\n  Round-trip error: {np.max(np.abs(R - R_from_q)):.2e}")

**Key observations:**
- Conversions are **exact** (up to floating-point precision), they describe the same rotation.
- Euler angle extraction has an inherent ambiguity: multiple angle triples can produce the same matrix.
- In practice, **store** rotations as quaternions or matrices; **display** them as Euler angles for human readability.

## 4.5 Interpolation and Stability

When planning smooth trajectories, we need to **interpolate between orientations**.
Linearly interpolating Euler angles produces ugly, non-uniform motion.
Quaternion **SLERP** (Spherical Linear Interpolation) gives constant angular velocity:

$$\text{SLERP}(q_0, q_1, t) = \frac{\sin((1-t)\Omega)}{\sin\Omega} q_0 + \frac{\sin(t\Omega)}{\sin\Omega} q_1$$

where $\Omega = \arccos(q_0 \cdot q_1)$.

### Example: Camera rotation between two viewpoints

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
# Start orientation (Euler degrees)
roll1, pitch1, yaw1 = 0.0, 0.0, 0.0
# End orientation (Euler degrees)
roll2, pitch2, yaw2 = 45.0, 60.0, 90.0
n_steps = 10     # number of interpolation steps  (try 5, 10, 20)
# ──────────────────────────────────────────────────────────────────────────────

q_start = quat_from_R(euler_to_R(np.radians(roll1), np.radians(pitch1), np.radians(yaw1)))
q_end = quat_from_R(euler_to_R(np.radians(roll2), np.radians(pitch2), np.radians(yaw2)))

euler_start = np.array([roll1, pitch1, yaw1])
euler_end = np.array([roll2, pitch2, yaw2])

fig = plt.figure(figsize=(16, 6))
t_values = np.linspace(0, 1, n_steps)

# SLERP interpolation
ax1 = fig.add_subplot(121, projection='3d')
ax1.set_title("Quaternion SLERP (smooth)", fontsize=13)
for i, t in enumerate(t_values):
    q_t = slerp(q_start, q_end, t)
    R_t = quat_to_R(q_t)
    offset = np.array([0, 0, 0])  # all at origin
    alpha = 0.3 + 0.7 * t
    draw_frame_3d(ax1, R_t, origin=offset, length=0.6)
ax1.set_xlim(-1.2, 1.2); ax1.set_ylim(-1.2, 1.2); ax1.set_zlim(-1.2, 1.2)
ax1.set_xlabel("X"); ax1.set_ylabel("Y"); ax1.set_zlabel("Z")

# Euler linear interpolation
ax2 = fig.add_subplot(122, projection='3d')
ax2.set_title("Euler linear interp (NOT smooth)", fontsize=13)
for i, t in enumerate(t_values):
    euler_t = (1 - t) * euler_start + t * euler_end
    R_t = euler_to_R(np.radians(euler_t[0]), np.radians(euler_t[1]), np.radians(euler_t[2]))
    draw_frame_3d(ax2, R_t, origin=np.array([0,0,0]), length=0.6)
ax2.set_xlim(-1.2, 1.2); ax2.set_ylim(-1.2, 1.2); ax2.set_zlim(-1.2, 1.2)
ax2.set_xlabel("X"); ax2.set_ylabel("Y"); ax2.set_zlabel("Z")

plt.tight_layout()
plt.show()

# Plot angular velocity profile
angles_slerp = []
angles_euler = []
ts = np.linspace(0, 1, 50)
for t in ts:
    q_t = slerp(q_start, q_end, t)
    R_t = quat_to_R(q_t)
    _, ang = R_to_axis_angle(R_t)
    angles_slerp.append(np.degrees(ang))

    e_t = (1-t)*euler_start + t*euler_end
    R_e = euler_to_R(np.radians(e_t[0]), np.radians(e_t[1]), np.radians(e_t[2]))
    _, ang_e = R_to_axis_angle(R_e)
    angles_euler.append(np.degrees(ang_e))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ts, angles_slerp, 'steelblue', lw=2, label='SLERP (quaternion)')
ax.plot(ts, angles_euler, 'tomato', lw=2, ls='--', label='Euler linear interp')
ax.set_xlabel("t (interpolation parameter)"); ax.set_ylabel("Rotation magnitude (degrees)")
ax.set_title("Angular distance from identity over interpolation")
ax.legend(); plt.tight_layout(); plt.show()

**Key observations:**
- SLERP follows the **shortest path** on the rotation manifold (the 3-sphere $S^3$).
- Euler linear interpolation can overshoot, wobble, or take indirect paths.
- For robotics trajectory planning, **always use SLERP** (or squad for splines), never interpolate Euler angles.

## 4.6 Choosing Representations in Practice

| Property | Euler Angles | Axis-Angle | Quaternion | Rotation Matrix |
|----------|:---:|:---:|:---:|:---:|
| **Parameters** | 3 | 3+1 | 4 | 9 |
| **Singularities** | Yes (gimbal lock) | $\theta = 0$ | None | None |
| **Composition** | Difficult | Difficult | $q_1 \otimes q_2$ | $R_1 R_2$ |
| **Interpolation** | Bad | OK | SLERP (best) | Bad |
| **Memory** | 3 floats | 4 floats | 4 floats | 9 floats |
| **Typical use** | Human I/O, display | Optimization, Lie algebra | State estimation, VIO | Forward kinematics, rendering |

### Example: 3D LiDAR point cloud rotation

A 3D LiDAR scan contains thousands of points. Rotating the entire cloud is a single matrix multiplication, or, equivalently, quaternion rotation of each point.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_ground = 300      # ground plane points
n_wall = 100        # vertical wall points
n_poles = 50        # pole/pillar points
yaw_deg = 30.0      # rotation to apply (degrees)
pitch_deg = 10.0
# ──────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

# Generate a realistic-ish 3D LiDAR scene
# Ground plane (z ≈ 0)
ground = np.column_stack([np.random.uniform(-10, 10, n_ground),
                          np.random.uniform(-10, 10, n_ground),
                          np.random.normal(0, 0.05, n_ground)])
# Wall (y ≈ 8)
wall = np.column_stack([np.random.uniform(-5, 5, n_wall),
                        np.full(n_wall, 8) + np.random.normal(0, 0.1, n_wall),
                        np.random.uniform(0, 3, n_wall)])
# Poles
pole1 = np.column_stack([np.random.normal(3, 0.1, n_poles),
                         np.random.normal(4, 0.1, n_poles),
                         np.random.uniform(0, 2.5, n_poles)])
pole2 = np.column_stack([np.random.normal(-4, 0.1, n_poles),
                         np.random.normal(5, 0.1, n_poles),
                         np.random.uniform(0, 2, n_poles)])

scan = np.vstack([ground, wall, pole1, pole2])

# Rotate using quaternion
q_rot = quat_from_R(euler_to_R(0, np.radians(pitch_deg), np.radians(yaw_deg)))
scan_rotated = np.array([quat_rotate(q_rot, p) for p in scan])

fig = plt.figure(figsize=(16, 6))

ax1 = fig.add_subplot(121, projection='3d')
ax1.set_title("Original 3D LiDAR scan", fontsize=13)
ax1.scatter(scan[:, 0], scan[:, 1], scan[:, 2], c=scan[:, 2], cmap='viridis', s=2, alpha=0.6)
draw_frame_3d(ax1, np.eye(3), length=2)

ax2 = fig.add_subplot(122, projection='3d')
ax2.set_title(f"Rotated (yaw={yaw_deg:.0f}°, pitch={pitch_deg:.0f}°)", fontsize=13)
ax2.scatter(scan_rotated[:, 0], scan_rotated[:, 1], scan_rotated[:, 2],
            c=scan_rotated[:, 2], cmap='magma', s=2, alpha=0.6)
R_applied = quat_to_R(q_rot)
draw_frame_3d(ax2, R_applied, length=2)

for ax in [ax1, ax2]:
    ax.set_xlim(-12, 12); ax.set_ylim(-12, 12); ax.set_zlim(-2, 5)
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")

plt.tight_layout()
plt.show()

print(f"Scan: {scan.shape[0]} points")
print(f"Quaternion used: [{q_rot[0]:.4f}, {q_rot[1]:.4f}, {q_rot[2]:.4f}, {q_rot[3]:.4f}]")

**Key observations:**
- For **large point clouds**, using the rotation matrix ($R \mathbf{p}$) is faster than quaternion rotation ($q p q^*$).
- Quaternions shine in **state estimation** where you need smooth updates, interpolation, and no singularities.
- In SLAM systems: poses are often stored as $(x, y, z, q_w, q_x, q_y, q_z)$, 7 numbers for a full 6-DOF pose.

---

## Exercises

### Exercise 4.1: Verify ZYX Euler rotation matrix

Implement $R_x(\phi)$, $R_y(\theta)$, $R_z(\psi)$ from scratch and verify that
$R_z(\psi) R_y(\theta) R_x(\phi)$ matches the `euler_to_R` function for
$(\phi, \theta, \psi) = (20°, 35°, -50°)$.

In [ ]:
# Your code here
phi, theta, psi = np.radians(20), np.radians(35), np.radians(-50)

# Build Rx, Ry, Rz manually and multiply
# Compare with euler_to_R(phi, theta, psi)

### Exercise 4.2: Relative rotation

Two frames $\{A\}$ and $\{B\}$ have orientations given by:
- $\{A\}$: roll=10°, pitch=20°, yaw=30°
- $\{B\}$: roll=5°, pitch=50°, yaw=-10°

Compute ${}^{A}_{B}R$ (rotation from $\{B\}$ to $\{A\}$), then extract its axis-angle representation.
What single rotation takes $\{A\}$ to $\{B\}$?

In [ ]:
# Your code here
R_A = euler_to_R(np.radians(10), np.radians(20), np.radians(30))
R_B = euler_to_R(np.radians(5), np.radians(50), np.radians(-10))

# Hint: R_A_B = R_A.T @ R_B  (relative rotation)
# Then use R_to_axis_angle

### Exercise 4.3: Implement SLERP from scratch

Without using the `slerp` function above, implement your own SLERP.
Test it by interpolating between $q_0 = [1, 0, 0, 0]$ (identity) and
$q_1 = $ quaternion for 90° about Z, with 5 intermediate steps.
Print the Euler angles at each step, they should change uniformly.

In [ ]:
# Your code here
q0 = np.array([1.0, 0, 0, 0])
q1 = quat_from_axis_angle(np.array([0, 0, 1]), np.radians(90))

# Implement SLERP: use np.arccos(dot), np.sin, etc.
# for t in [0, 0.25, 0.5, 0.75, 1.0]:
#     q_t = my_slerp(q0, q1, t)
#     R_t = quat_to_R(q_t)
#     print(R_to_euler(R_t))

### Exercise 4.4: IMU orientation update

A drone's IMU reports orientation as Euler angles and gyroscope readings as angular velocities.

Starting orientation: roll=0°, pitch=0°, yaw=0°.
Gyroscope reading: $\omega = [0.1, 0.05, 0.2]$ rad/s.
Time step: $\Delta t = 0.1$ s.

Update the orientation using quaternions:
1. Convert current orientation to quaternion
2. Create a small rotation quaternion from $\omega \Delta t$
3. Compose: $q_{new} = q_{old} \otimes q_{delta}$
4. Convert back to Euler for display

Repeat for 20 time steps and plot roll, pitch, yaw over time.

In [ ]:
# Your code here
dt = 0.1
omega = np.array([0.1, 0.05, 0.2])  # rad/s
q = np.array([1.0, 0, 0, 0])  # start at identity

# Hint: for each step:
#   delta_angle = np.linalg.norm(omega * dt)
#   delta_axis = omega / np.linalg.norm(omega)
#   q_delta = quat_from_axis_angle(delta_axis, delta_angle)
#   q = quat_multiply(q, q_delta)
#   q = quat_normalize(q)

### Exercise 4.5: Multi-scan 3D registration (challenge)

Generate 3 synthetic LiDAR scans (each with 100 random 3D points).
Apply known rotations between consecutive scans:
- Scan 1 → Scan 2: 15° yaw
- Scan 2 → Scan 3: 10° yaw + 5° pitch

Transform all scans to the global frame (Scan 1 = global) and plot them together.
Use different colors for each scan.

In [ ]:
# Your code here
np.random.seed(123)
scan1 = np.random.randn(100, 3) * np.array([3, 3, 1])  # 100 points

# R_12 = rotation from scan 1 to scan 2 frame
# R_23 = rotation from scan 2 to scan 3 frame
# scan2_global = R_12 @ scan2_local.T
# scan3_global = R_12 @ R_23 @ scan3_local.T